In [4]:
import dashscope
from dashscope import Generation

dashscope.base_http_api_url = f"https://ws-05fne6an5d21rmhd.cn-beijing.maas.aliyuncs.com/api/v1"
# 2. 填入欧洲空间创建的sk密钥
dashscope.api_key = "sk-ws-H.EMXMXHY.hVls.MEUCIFVorhP99Nz-f3uvoKRnBTntkHPT_CqSKIruOfhqSPPhAiEAu52Y31ZZXeIoOcPWxcglHj9spSJ9okCEcrqjpnYmRvg"

resp = Generation.call(
    model="qwen-turbo",
    prompt="你好",
    timeout=120
)

# 打印完整返回结构，看报错信息
print("完整返回对象：")
print(resp)
print("\n状态码：", resp.status_code)
print("错误码code：", resp.code)
print("错误信息message：", resp.message)
print("输出output：", resp.output)


完整返回对象：
{"status_code": 200, "request_id": "7f30df71-4b8a-974f-a30a-5c1e40221326", "code": "", "message": "", "output": {"text": "你好！很高兴见到你！😊 今天过得怎么样呀？有什么我可以帮你的吗？", "finish_reason": "stop", "choices": null}, "usage": {"input_tokens": 13, "output_tokens": 19, "total_tokens": 32, "prompt_tokens_details": {"cached_tokens": 0}}}

状态码： HTTPStatus.OK
错误码code： 
错误信息message： 
输出output： {"text": "你好！很高兴见到你！😊 今天过得怎么样呀？有什么我可以帮你的吗？", "finish_reason": "stop", "choices": null}


In [1]:
import os
import dashscope
from dashscope import Generation
import json
import time
dashscope.base_http_api_url = f"https://ws-05fne6an5d21rmhd.cn-beijing.maas.aliyuncs.com/api/v1"
# 2. 填入欧洲空间创建的sk密钥
dashscope.api_key = "sk-ws-H.EMXMXHY.hVls.MEUCIFVorhP99Nz-f3uvoKRnBTntkHPT_CqSKIruOfhqSPPhAiEAu52Y31ZZXeIoOcPWxcglHj9spSJ9okCEcrqjpnYmRvg"

TXT_PATH = "book/卡拉马佐夫兄弟.txt"
OUT_JSON_PATH = "karamazov_qa.json"
DISTILL_JSONL = "distill_train.jsonl"
CHUNK_LENGTH = 800  # 缩小文本块，减少超限
RETRY_TIMES = 2     # 失败重试2次
SLEEP_SEC = 1.5     # 每次调用间隔，防限流
# ==========================

def build_qa_from_text(text_segment):
    sys_prompt = """
根据《卡拉马佐夫兄弟》原文段落，生成3-5组问答，答案严格取自原文。
仅输出纯JSON数组，不要任何额外文字、代码块标记、解释。
格式示例：
[
    {"question": "问题", "answer": "回答"}
]
"""
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": f"原文段落：\n{text_segment}"}
    ]

    # 重试逻辑
    for retry in range(RETRY_TIMES + 1):
        try:
            resp = Generation.call(
                model="qwen-plus",
                messages=messages,
                result_format="message",
                timeout=90
            )
            # 关键校验：防止output为空
            if not hasattr(resp, "output") or resp.output is None:
                raise Exception("接口返回output为空")
            if not hasattr(resp.output, "choices") or len(resp.output.choices) == 0:
                raise Exception("choices不存在")

            content = resp.output.choices[0].message.content.strip()
            qa_list = json.loads(content)
            return qa_list

        except Exception as err:
            if retry < RETRY_TIMES:
                print(f"重试{retry+1}次，错误：{str(err)}")
                time.sleep(SLEEP_SEC * 2)
            else:
                raise err

def read_file_by_chunk(file_path, chunk_size):
    with open(file_path, "r", encoding="utf-8") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            yield chunk

def convert_qa_for_distillation(qa_list):
    data = []
    for item in qa_list:
        data.append({
            "messages": [
                {"role": "system", "content": "你是《卡拉马佐夫兄弟》专业文学解读助手，回答准确贴合原著内容。"},
                {"role": "user", "content": item["question"]},
                {"role": "assistant", "content": item["answer"]}
            ]
        })
    return data

if __name__ == "__main__":
    all_qa = []
    for idx, seg in enumerate(read_file_by_chunk(TXT_PATH, CHUNK_LENGTH)):
        print(f"正在处理第 {idx+1} 段文本...")
        try:
            batch = build_qa_from_text(seg)
            all_qa.extend(batch)
            print(f"本段生成 {len(batch)} 组问答")
        except Exception as e:
            print(f"第{idx+1}段处理失败：{str(e)}")
        # 每段调用后休眠，降低接口压力
        time.sleep(SLEEP_SEC)

    # 保存原始QA
    with open(OUT_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(all_qa, f, ensure_ascii=False, indent=2)
    # 转换蒸馏训练集jsonl
    distill_data = convert_qa_for_distillation(all_qa)
    with open(DISTILL_JSONL, "w", encoding="utf-8") as f:
        for line in distill_data:
            json.dump(line, f, ensure_ascii=False)
            f.write("\n")
    print(f"全部处理完成！总问答样本：{len(all_qa)}")
    print(f"原始问答文件：{OUT_JSON_PATH}")
    print(f"蒸馏微调数据集：{DISTILL_JSONL}")


正在处理第 1 段文本...
本段生成 5 组问答
正在处理第 2 段文本...
本段生成 5 组问答
正在处理第 3 段文本...
本段生成 5 组问答
正在处理第 4 段文本...
本段生成 5 组问答
正在处理第 5 段文本...
本段生成 5 组问答
正在处理第 6 段文本...
本段生成 5 组问答
正在处理第 7 段文本...
本段生成 5 组问答
正在处理第 8 段文本...
本段生成 5 组问答
正在处理第 9 段文本...
本段生成 5 组问答
正在处理第 10 段文本...
本段生成 5 组问答
正在处理第 11 段文本...
本段生成 5 组问答
正在处理第 12 段文本...
本段生成 5 组问答
正在处理第 13 段文本...
本段生成 5 组问答
正在处理第 14 段文本...
本段生成 5 组问答
正在处理第 15 段文本...
本段生成 5 组问答
正在处理第 16 段文本...
本段生成 5 组问答
正在处理第 17 段文本...
本段生成 5 组问答
正在处理第 18 段文本...
本段生成 5 组问答
正在处理第 19 段文本...
本段生成 5 组问答
正在处理第 20 段文本...
本段生成 5 组问答
正在处理第 21 段文本...
本段生成 5 组问答
正在处理第 22 段文本...
本段生成 5 组问答
正在处理第 23 段文本...
本段生成 5 组问答
正在处理第 24 段文本...
本段生成 5 组问答
正在处理第 25 段文本...
本段生成 5 组问答
正在处理第 26 段文本...
本段生成 4 组问答
正在处理第 27 段文本...
本段生成 5 组问答
正在处理第 28 段文本...
本段生成 5 组问答
正在处理第 29 段文本...
本段生成 5 组问答
正在处理第 30 段文本...
本段生成 5 组问答
正在处理第 31 段文本...
本段生成 5 组问答
正在处理第 32 段文本...
本段生成 5 组问答
正在处理第 33 段文本...
本段生成 5 组问答
正在处理第 34 段文本...
本段生成 5 组问答
正在处理第 35 段文本...
本段生成 5 组问答
正在处理第 36 段文本...
本段生成 5 组问答
正在处理第 37 段文本...
本段生成 5 组问答
正在处理第 38 段

In [ ]:
print("ok")

In [3]:
import json
from collections import Counter

file_path = "distill_train.jsonl"
stats = Counter()
error_lines = []

with open(file_path, "r", encoding="utf-8") as f:
    for idx, raw in enumerate(f, 1):
        line = raw.strip()
        if not line:
            error_lines.append(f"第{idx}行：空行")
            stats["空行"] += 1
            continue
        # 校验JSON合法性
        try:
            item = json.loads(line)
        except Exception as e:
            error_lines.append(f"第{idx}行：JSON解析失败 {str(e)}")
            stats["非法JSON"] += 1
            continue
        # 校验messages结构
        if "messages" not in item or len(item["messages"]) != 3:
            error_lines.append(f"第{idx}行：messages缺失/角色数量不对")
            stats["格式错误"] += 1
            continue
        roles = [m["role"] for m in item["messages"]]
        if roles != ["system", "user", "assistant"]:
            error_lines.append(f"第{idx}行：角色顺序错误")
            stats["角色错误"] += 1
            continue
        # 过滤空问答、过短/过长回答
        user_text = item["messages"][1]["content"].strip()
        ans_text = item["messages"][2]["content"].strip()
        if len(user_text) < 4 or len(ans_text) < 3:
            error_lines.append(f"第{idx}行：问答内容过短")
            stats["低质样本"] += 1
            continue
        stats["有效样本"] += 1

print("统计结果：", dict(stats))
if error_lines:
    print("错误样本列表：")
    for e in error_lines[:20]:
        print(e)


统计结果： {'有效样本': 4300, '低质样本': 76}
错误样本列表：
第12行：问答内容过短
第14行：问答内容过短
第286行：问答内容过短
第502行：问答内容过短
第529行：问答内容过短
第530行：问答内容过短
第544行：问答内容过短
第723行：问答内容过短
第847行：问答内容过短
第982行：问答内容过短
第984行：问答内容过短
第1003行：问答内容过短
第1022行：问答内容过短
第1086行：问答内容过短
第1136行：问答内容过短
第1138行：问答内容过短
第1191行：问答内容过短
第1228行：问答内容过短
第1229行：问答内容过短
第1271行：问答内容过短


In [4]:
import json
import random
from sklearn.model_selection import train_test_split

all_data = []
with open("distill_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        all_data.append(json.loads(line))

# 随机划分
train, val = train_test_split(all_data, test_size=0.1, random_state=42)

# 保存文件
def save_jsonl(data, save_name):
    with open(save_name, "w", encoding="utf-8") as f:
        for d in data:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

save_jsonl(train, "train.jsonl")
save_jsonl(val, "val.jsonl")
print(f"训练集{len(train)}条，验证集{len(val)}条")


训练集3938条，验证集438条
